In [24]:
import time, numpy as np
from openpi_client import websocket_client_policy, image_tools
from droid_plus.robot import DroidPlus

POLICY_HOST, POLICY_PORT = "localhost", 18000
PROMPT        = "pick up the cube"      # <-- your task string

CONTROL_HZ    = 15.0                       # must stay >= 10 (franky watchdog = 100 ms)
EXECUTE_STEPS = 8                          # steps run per chunk before re-inferring (<=15)
MAX_CHUNKS    = 100
VEL_CAP       = 0.25                       # rad/s, hard per-joint clip on policy output
MAX_CHUNK_DISP= 0.5                       # rad, abort a chunk that integrates past this on any joint
CMD_TIMEOUT_S = 3.0

# gripper
GRIPPER_MODE  = "auto"                     # "auto" (policy-driven) | "manual" | "off"
GRIPPER_STYLE = "binary"                   # "binary" (hysteresis) | "continuous"
CLOSE_TH, OPEN_TH = 0.55, 0.45
GRIP_SPEED, GRIP_FORCE = 120, 80           # ~0.047 m/s, ~16 N
APPROVE_FIRST_CLOSE = True

DRY_RUN = False                             # True = print only, no arm/gripper commands

# FR3 joint limits (verify against your datasheet)
Q_MIN = np.array([-2.7437,-1.7837,-2.9007,-3.0421,-2.8065, 0.5445,-3.0159])
Q_MAX = np.array([ 2.7437, 1.7837, 2.9007,-0.1518, 2.8065, 4.5169, 3.0159])
MARGIN = 0.12
START_Q = np.array([0.0, 0.0, 0.0, -1.571, 0.0, 1.571, 0.0])   # your teleop data starts every episode here
DT = 1.0 / CONTROL_HZ


In [25]:
droid  = DroidPlus()
g      = droid.gripper
policy = websocket_client_policy.WebsocketClientPolicy(POLICY_HOST, POLICY_PORT)
print("server meta:", policy.get_server_metadata())

def get_obs():
    left  = droid.get_left_image(jpeg_quality=90)
    wrist = droid.get_wrist_image(jpeg_quality=90)
    q     = np.asarray(droid.get_current_joint_state()["positions"], dtype=np.float32)
    closed = float(g.gripper_position_frac())          # 0=open, 1=closed
    return left, wrist, q, closed

def infer(left, wrist, q, closed):
    obs = {
        "observation/exterior_image_1_left": image_tools.convert_to_uint8(image_tools.resize_with_pad(left, 224, 224)),
        "observation/wrist_image_left":      image_tools.convert_to_uint8(image_tools.resize_with_pad(wrist, 224, 224)),
        "observation/joint_position":        q.astype(np.float32),
        "observation/gripper_position":      np.asarray([closed], dtype=np.float32),
        "prompt": PROMPT,
    }
    return np.asarray(policy.infer(obs)["actions"], dtype=float)   # (15, 8): [:, :7]=rad/s, [:, 7]=closedness

def guard_chunk(q0, v):
    q_traj = q0 + np.cumsum(v * DT, axis=0)
    lo, hi = Q_MIN + MARGIN, Q_MAX - MARGIN
    if (q_traj < lo).any() or (q_traj > hi).any():
        bad = np.where((q_traj < lo) | (q_traj > hi))[1]
        raise RuntimeError(f"chunk exits guarded joint limits on joints {sorted(set(bad))}")
    disp = np.abs(q_traj[-1] - q0).max()
    if disp > MAX_CHUNK_DISP:
        raise RuntimeError(f"chunk displacement {disp:.3f} rad > MAX_CHUNK_DISP")
    return q_traj

def send_q(q, seq):
    if not DRY_RUN:
        droid.set_target_joint_state(np.clip(q, Q_MIN, Q_MAX), velocities=[0.0]*7, seq=seq)

def soft_stop():
    time.sleep(0.15)            # let the 100 ms velocity watchdog decay to zero first
    try: droid.stop()
    except Exception: pass

_grip = {"closed": None}       # None=unknown, True=closed, False=open
def command_gripper(closedness_plan, chunk):
    if GRIPPER_MODE != "auto" or DRY_RUN:
        return
    want = float(np.clip(np.max(closedness_plan), 0.0, 1.0))
    if GRIPPER_STYLE == "continuous":
        g.go_to_async(int(round(want * 255)), speed=GRIP_SPEED, force=GRIP_FORCE)
        return
    if want >= CLOSE_TH and _grip["closed"] is not True:
        if APPROVE_FIRST_CLOSE and _grip["closed"] is None:
            input(f"[chunk {chunk}] policy wants CLOSE — Enter to allow, Ctrl+C to abort ")
        g.close_async(speed=GRIP_SPEED, force=GRIP_FORCE); _grip["closed"] = True
        print(f"   gripper -> CLOSE")
    elif want <= OPEN_TH and _grip["closed"] is not False:
        g.open_async(speed=GRIP_SPEED); _grip["closed"] = False
        print(f"   gripper -> OPEN")


gripper initialized None
server meta: {}


In [26]:
if not DRY_RUN:
    g.open(speed=GRIP_SPEED)
    droid.robot.set_command_timeout(CMD_TIMEOUT_S)

sent = get_obs()[2].astype(float)
print("current q:", np.round(sent, 3), " -> START_Q:", START_Q)
input("workspace clear, E-stop in hand — Enter to reset arm to start pose ")

seq = 0
try:
    while np.abs(START_Q - sent).max() > 1e-3:
        sent = np.clip(sent + np.clip(START_Q - sent, -0.03, 0.03), Q_MIN, Q_MAX)  # ~0.45 rad/s
        send_q(sent, seq); seq += 1; time.sleep(DT)
    for _ in range(8):                       # hold so watchdog doesn't cut it mid-settle
        send_q(sent, seq); seq += 1; time.sleep(DT)
finally:
    soft_stop()
print("at start:", np.round(get_obs()[2], 3))


current q: [ 0.2    0.507 -0.138 -1.116  0.029  1.614  0.168]  -> START_Q: [ 0.     0.     0.    -1.571  0.     1.571  0.   ]
at start: [-0.    -0.002 -0.    -1.573 -0.     1.571 -0.   ]


In [27]:
if not DRY_RUN:
    droid.robot.set_command_timeout(CMD_TIMEOUT_S)
_grip["closed"] = None
seq = 1000
try:
    for chunk in range(1, MAX_CHUNKS + 1):
        left, wrist, q, closed = get_obs()
        actions = infer(left, wrist, q, closed)

        v      = np.clip(actions[:EXECUTE_STEPS, :7], -VEL_CAP, VEL_CAP)
        gplan  = actions[:EXECUTE_STEPS, 7]
        q_traj = guard_chunk(q, v)

        print(f"[{chunk:02d}/{MAX_CHUNKS}] q_disp={np.abs(q_traj[-1]-q).max():.3f} rad "
              f"vpeak={np.abs(actions[:EXECUTE_STEPS,:7]).max():.2f} "
              f"grip={gplan.min():.2f}..{gplan.max():.2f} closed_now={closed:.2f}"
              + ("   [DRY]" if DRY_RUN else ""))

        command_gripper(gplan, chunk)

        t = time.time()
        for qk in q_traj:
            send_q(qk, seq); seq += 1
            t += DT
            time.sleep(max(0.0, t - time.time()))
    else:
        print("MAX_CHUNKS reached")
except KeyboardInterrupt:
    print("interrupted by user")
except Exception as e:
    print("ABORT:", type(e).__name__, e)
finally:
    soft_stop()
    print("stopped. final q:", np.round(get_obs()[2], 3))


[01/100] q_disp=0.131 rad vpeak=0.31 grip=0.01..0.02 closed_now=0.00
   gripper -> OPEN
[02/100] q_disp=0.133 rad vpeak=0.31 grip=0.01..0.01 closed_now=0.00
[03/100] q_disp=0.132 rad vpeak=0.27 grip=0.01..0.02 closed_now=0.00
[04/100] q_disp=0.058 rad vpeak=0.28 grip=0.01..0.01 closed_now=0.00
[05/100] q_disp=0.031 rad vpeak=0.32 grip=0.01..0.02 closed_now=0.00
[06/100] q_disp=0.049 rad vpeak=0.22 grip=0.01..0.01 closed_now=0.00
[07/100] q_disp=0.028 rad vpeak=0.16 grip=0.01..0.01 closed_now=0.00
[08/100] q_disp=0.077 rad vpeak=0.26 grip=0.01..0.01 closed_now=0.00
[09/100] q_disp=0.040 rad vpeak=0.26 grip=0.01..0.01 closed_now=0.00
[10/100] q_disp=0.003 rad vpeak=0.03 grip=0.01..0.01 closed_now=0.00
[11/100] q_disp=0.010 rad vpeak=0.12 grip=0.01..0.01 closed_now=0.00
[12/100] q_disp=0.004 rad vpeak=0.03 grip=0.01..0.02 closed_now=0.00
[13/100] q_disp=0.105 rad vpeak=0.23 grip=0.01..0.01 closed_now=0.00
[14/100] q_disp=0.060 rad vpeak=0.21 grip=0.01..0.02 closed_now=0.00
[15/100] q_disp